[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/04-recall-tuning/04-reusable_recall_configs_as_json.ipynb)

# Reusable Recall Configs as JSON

A `TableRecallConfig` you build in a notebook lives only in that notebook's memory. Every weight, every `minimum_quality` floor, every mode you carefully tuned across the last three notebooks disappears the moment the kernel restarts, unless you rebuild it from scratch.

Just like `TableConfig` in the previous directory, `TableRecallConfig` supports `to_json()` and `from_json()`, so a fully tuned recall strategy can be saved once, version-controlled, and loaded anywhere that needs to search the same way.

In this notebook you will:

1. Build a complete, multi-field recall configuration worth saving
2. Save it to a JSON file and look at what actually gets written
3. Load it back in a simulated separate process
4. Confirm the loaded configuration produces identical matching behavior
5. See the dictionary-based alternative for cases where a file on disk is not the right fit
6. Pick up a few practical habits for managing recall configs over time

**Estimated time:** 10 minutes
**Difficulty:** Intermediate
**Prerequisites:** `02-weighting_fields_for_better_precision.ipynb`, `03-numeric_matching_ranges_and_thresholds.ipynb`


In [ ]:
# !pip install mbox

## 1. Build a recall configuration worth saving

Let's combine everything from the last two notebooks into one configuration: a weighted text field, and a numeric field with a threshold mode.

In [ ]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.config import TableConfig, TableFieldConfig, IndexType
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

df = pd.DataFrame({
    "product_name": ["Extended Battery Pack", "Portable Power Bank", "Motion Sensor Camera", "Smart Relay Switch", "Solar Panel Charger"],
    "unit_price": [24.99, 39.50, 59.00, 18.75, 89.99]
})

schema = TableConfig(fields=[
    TableFieldConfig(column="product_name", index_type=IndexType.PHRASE),
    TableFieldConfig(column="unit_price", index_type=IndexType.DOUBLE)
])

index = TableIndexer.create_index(
    df=df,
    config_overrides=schema,
    tmp_dir="tmp_index"
)

budget_search_config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(
            input_column="product_name",
            indexed_column="product_name",
            minimum_quality=40,
            weight=60,
            mode=TableRecallMode.APPROX
        ),
        TableRecallFieldConfig(
            input_column="unit_price",
            indexed_column="unit_price",
            minimum_quality=0,
            weight=40,
            mode=TableRecallMode.NUM_LOWER
        )
    ],
    max_results=5,
    min_total_match_value=0,
    include_field_scores=True
)

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object
config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


Let's confirm it works before saving anything.

In [2]:
verification = index.match(
    queries=pd.DataFrame({"product_name": ["Battery Pack"], "unit_price": [30.00]}),
    config=budget_search_config
)

verification

,query_row,index_row,product_name_candidate,unit_price_candidate,overall_score,product_name_score,unit_price_score
0,0,0,Extended Battery Pack,24.99,95,92,100


## 2. Save the configuration to JSON

`to_json()` writes the complete configuration, every field's weight, mode, and quality floor, plus the global settings, to a single file.

In [3]:
import os

os.makedirs("configs", exist_ok=True)
budget_search_config.to_json("./configs/budget_search_config.json")
print("Config saved.")

Config saved.


Let's look at what actually got written.

In [4]:
with open("./configs/budget_search_config.json") as f:
    print(f.read())

{"fields": [{"input_column": "product_name", "indexed_column": "product_name", "minimum_quality": 40, "weight": 60, "mode": "Approx"}, {"input_column": "unit_price", "indexed_column": "unit_price", "minimum_quality": 0, "weight": 40, "mode": "Detect"}], "max_results": 5, "max_quality_spread": 100, "min_total_match_value": 0, "non_search_output_fields": [], "include_queries": false, "include_total_score": true, "include_field_scores": true}


This file is plain, readable JSON, every field's `weight`, `mode`, and `minimum_quality`, alongside the global `max_results` and `min_total_match_value`. If a field also had a `character_mapping` or `aliases` attached, those would appear nested here too, the same way they did in the schema JSON from the previous directory. It is exactly the kind of file that belongs in version control and in a pull request review, not something generated silently at runtime and never looked at again.

## 3. Load the configuration in a simulated separate process

To make the point clearly, delete the configuration object entirely, so nothing but the JSON file on disk remains, then load it back.

In [5]:
del budget_search_config

loaded_config = TableRecallConfig.from_json("./configs/budget_search_config.json")
loaded_config

TableRecallConfig(fields=[TableRecallFieldConfig(input_column='product_name', indexed_column='product_name', minimum_quality=40, weight=60, mode=<TableRecallMode.APPROX: 'Approx'>), TableRecallFieldConfig(input_column='unit_price', indexed_column='unit_price', minimum_quality=0, weight=40, mode=<TableRecallMode.DETECT: 'Detect'>)], max_results=5, max_quality_spread=100, min_total_match_value=0, non_search_output_fields=[], include_queries=False, include_total_score=True, include_field_scores=True)

## 4. Confirm it behaves identically

Run the exact same query from Step 1 against the loaded configuration.

In [6]:
loaded_results = index.match(
    queries=pd.DataFrame({"product_name": ["Battery Pack"], "unit_price": [30.00]}),
    config=loaded_config
)

loaded_results

,query_row,index_row,product_name_candidate,unit_price_candidate,overall_score,product_name_score,unit_price_score
0,0,0,Extended Battery Pack,24.99,95,92,100


This should produce the same ranking and the same scores as Step 1's verification query. Nothing about the weighting, the numeric threshold, or the quality floor was lost by writing the configuration to disk and loading it back somewhere else.

## 5. When you need a dict instead of a file

The same interoperability pattern from schema serialization applies here. `to_dict()` and `from_dict()` cover cases where a recall configuration needs to travel somewhere that is not a filesystem path, an API request body, a message queue payload, a row in a settings table.

In [7]:
config_dict = loaded_config.to_dict()
print(type(config_dict))

rebuilt_config = TableRecallConfig.from_dict(config_dict)
rebuilt_config.fields[0].weight

<class 'dict'>


60

## 6. A few practical notes

**Keep index schemas and recall configs as separate files.** A `TableConfig` describes how data is structured and stored. A `TableRecallConfig` describes how a specific kind of search should behave. The same index can reasonably be searched with several different recall configs, a broad, permissive search for browsing, and a strict, high-precision search for deduplication, so keeping them as separate, independently versioned files usually pays off.

**Name recall config files by the search behavior they represent, not just the data they apply to.** `budget_search_config.json` says more about intent than `product_recall_config.json` does. If you build more than one recall strategy for the same index, distinct names prevent them from being confused later.

**Re-run your regression queries after loading a saved config, not just after building it.** The best-practices checklist from `02-weighting_fields_for_better_precision.ipynb`, testing weight changes against known-correct matches, applies just as much after a round trip through JSON as it does right after writing the weights the first time. A config that behaves correctly in memory should be reverified once it has been saved and reloaded, especially the first time you wire up the loading path in a new service.

**Treat a saved recall config as a snapshot.** If you tune `budget_search_config` further in a notebook but forget to call `to_json()` again, the file on disk still reflects the old version. There is no automatic sync between a live Python object and the file it was last saved to.

## Next steps

You now have the full recall-tuning toolkit: match modes, field weighting, numeric thresholds, and portable, version-controllable configuration. From here:

- **`05-detecting_phrases_in_free_text.ipynb`** - a deep dive on `DETECT` mode specifically, finding a short known phrase inside a longer, messier piece of text
- **`05-explainability/`** - understand exactly why a match scored what it did, going deeper into the field-level scores you have been reading throughout this directory

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*